# Creating Machine Learning Ready Datasets

**Important Note**: This notebook was updated on 03.18.26 in order to address the order in which we created the dataset. Prior to this change, we forward filled all data which included weekends and holidays. After the update, we now only forward fill the MacroEconomic data and then use the price data to drop non-trading days. Check version histories if desired.

The purpose of this notebook is to merge the macroeconomic dataset and the price data. We need to forward fill the data that comes infrequently. We then need to calculate the forward looking log retruns (by one quarter) and annualize it, and then calculate the forward looking volatility and annualize that. These vectors will be our labels.

After that, I want to create a second CSV that uses stationary data (differenced) instead of the raw data to see whether or not this imporves the performance of our LSTM. This dataset will be more similar to how financial analysts and economoists look at data, however, it is possible that deep learning will find similar patterns in the raw data because of the nature of deep learning. We will compare model performance with the two datasets.

## Import Libraries
Let's create a section for all of the libraries that will be necessary for this manipulation.

In [ ]:
# Manipulation libraries  
import numpy as np
import pandas as pd
import altair as alt

from pathlib import Path

# Deactivate the max rows and columns limit for Altair
alt.data_transformers.disable_max_rows()


# Import the datasets
Now, we can pull in both of the CSV files that we will merge together.

In [ ]:
base_dir = Path.cwd()

macro_df = pd.read_csv(base_dir/'fred_data.csv')
price_df = pd.read_csv(base_dir/'yfinance_data.csv')
print(f"Macro DataFrame shape: {macro_df.shape}, Price DataFrame shape: {price_df.shape}")

Now, let's take a look at the dataframes to make sure we can merge on the 'date'.

In [ ]:
macro_df.head()

In [ ]:
price_df.head()

This is the first large adaptation after running our first model where we artificially used data from weekends and holidays. Let's forward fill the macro data before merging.

In [ ]:
macro_filled_df = macro_df.ffill()
macro_filled_df.head(10)

Interestingly, we can see that we had some duplicate data (i.e. TLT) when we merged our data in the previous CSVs. So, let's loop through and keep only one of them.

In [ ]:
# Drop _y duplicates and rename _x back to original
x_cols = [col for col in price_df.columns if col.endswith('_x')]

for col in x_cols:
    base = col[:-2]  # strip '_x'
    price_df.drop(columns=[f'{base}_y'], inplace=True)
    price_df.rename(columns={col: base}, inplace=True)

Now, let's calculate all of our targets from the clean price dataframe before we merge them all together. This will be clean as there shouldn't be many NaN's except at the beginning

In [ ]:
target_variables = ['XLF','XLK','XLU','XLV','XLE','XLI','XLB','XLP','XLY','XLRE','BIL','IEF','TLT','LQD','HYG','TIP','GLD']

In [ ]:
for ticker in target_variables:
    mask = price_df[ticker].notna()
    ticker_series = price_df.loc[mask, ticker].copy()

    # Log return target — shift on clean trading-day series only
    log_returns = np.log(ticker_series.shift(-63) / ticker_series) * 4
    price_df.loc[mask, f'{ticker}_logreturn_target'] = log_returns.values

    # Volatility target — rolling on clean trading-day series only
    daily_lr = np.log(ticker_series / ticker_series.shift(1))
    vol = daily_lr.rolling(63).std() * np.sqrt(252)
    price_df.loc[mask, f'{ticker}_volatility_target'] = vol.shift(-63).values

Great, so this should be a simple horizontal merge of the dataframes on the 'date' column.

In [ ]:
merged_df = pd.merge(macro_filled_df, price_df, on='date', how='right')
merged_df.shape

Let's do a quick sanity check to see the minimum and maximum date.

In [ ]:
min, max = merged_df['date'].min(), merged_df['date'].max()
print(f"Date range: {min} to {max}")

Alright, this looks great. Let's pull all of the column names into a list so we can work with them a little easier.

In [ ]:
independent_columns = merged_df.columns.tolist()
print(f"Columns in merged DataFrame: {independent_columns}")

Now, let's forward fill the data so that less frequent data is filled into the dataframe so we have complete vectors at each timeframe. First, let's get an idea of the number of NaN's in each column, then forward fill, then double check to see if it is filling what we expect (like GDP)

In [ ]:
 merged_df.isna().sum()

In [ ]:
filled_df = merged_df.copy()

Alright, let's export this as a CSV that can now be worked on. It's important to note that this still has a significant number of NaN values and independent variables that we may end up dropping in our modelling but it is a workable dataset to start designing our machine learning pipeline around.

In [ ]:
#Let's confirm we don't have any weekends orholidays
filled_df.tail(100)

In [ ]:
filled_df.drop(columns=['DX-Y.NYB'], inplace=True)
filled_df.to_csv(base_dir/'raw_data_prediction_dataset.csv', index=False)

Let's take a quick look visually to see if these targets make sense.

In [ ]:
XLK_return_df = filled_df[['date', 'XLU_logreturn_target', 'XLK_logreturn_target']].copy().dropna()

# Mean lines
xlk_mean = XLK_return_df['XLK_logreturn_target'].mean()
xlu_mean = XLK_return_df['XLU_logreturn_target'].mean()

mean_df = pd.DataFrame({
    'date': [XLK_return_df['date'].min(), XLK_return_df['date'].max()],
    'xlk_mean': xlk_mean,
    'xlu_mean': xlu_mean
})

XLK_return = alt.Chart(XLK_return_df).mark_line().encode(
    x=alt.X('date:T', axis=alt.Axis(title='Date', grid=False)),
    y=alt.Y('XLK_logreturn_target:Q', axis=alt.Axis(title='Log Return', grid=False))
)
XLU_return = alt.Chart(XLK_return_df).mark_line(color='orange').encode(
    x='date:T',
    y='XLU_logreturn_target:Q'
)

xlk_mean_line = alt.Chart(mean_df).mark_line(strokeDash=[6,3]).encode(
    x='date:T',
    y='xlk_mean:Q'
)
xlu_mean_line = alt.Chart(mean_df).mark_line(strokeDash=[6,3], color='orange').encode(
    x='date:T',
    y='xlu_mean:Q'
)

(XLK_return + XLU_return + xlk_mean_line + xlu_mean_line).properties(
    title='XLK vs XLU Log Returns', height=400, width=900).configure_view(strokeWidth=0)

In [ ]:
XLK_volatility_df = filled_df[['date', 'XLU_volatility_target', 'XLK_volatility_target']].copy().dropna()

xlk_vol_mean = XLK_volatility_df['XLK_volatility_target'].mean()
xlu_vol_mean = XLK_volatility_df['XLU_volatility_target'].mean()

vol_mean_df = pd.DataFrame({
    'date': [XLK_volatility_df['date'].min(), XLK_volatility_df['date'].max()],
    'xlk_mean': xlk_vol_mean,
    'xlu_mean': xlu_vol_mean
})

XLK_volatility = alt.Chart(XLK_volatility_df).mark_line().encode(
    x=alt.X('date:T', axis=alt.Axis(title='Date', grid=False)),
    y=alt.Y('XLK_volatility_target:Q', axis=alt.Axis(title='Volatility', grid=False))
)
XLU_volatility = alt.Chart(XLK_volatility_df).mark_line(color='orange').encode(
    x='date:T',
    y='XLU_volatility_target:Q'
)

xlk_vol_mean_line = alt.Chart(vol_mean_df).mark_line(strokeDash=[6,3]).encode(
    x='date:T',
    y='xlk_mean:Q'
)
xlu_vol_mean_line = alt.Chart(vol_mean_df).mark_line(strokeDash=[6,3], color='orange').encode(
    x='date:T',
    y='xlu_mean:Q'
)

(XLK_volatility + XLU_volatility + xlk_vol_mean_line + xlu_vol_mean_line).properties(
    title='XLK vs XLU Volatilities', height=400, width=900).configure_view(strokeWidth=0)

In [ ]:
XLK_complete_df = filled_df[['date', 'XLK_logreturn_target', 'XLK_volatility_target']].copy().dropna()

base = alt.Chart(XLK_complete_df).encode(x=alt.X('date:T', axis=alt.Axis(title='Date', grid=False)))

XLK_volatility = base.mark_line().encode(
    y=alt.Y('XLK_volatility_target:Q', axis=alt.Axis(title='Volatility', grid=False, titleColor='steelblue', labelColor='steelblue'))
)

XLK_return = base.mark_line(color='red').encode(
    y=alt.Y('XLK_logreturn_target:Q', axis=alt.Axis(title='Log Return', grid=False, titleColor='red', labelColor='red'))
)

alt.layer(XLK_return,XLK_volatility).resolve_scale(
    y='independent'
).properties(
    title='XLK Log Returns and Volatility', height=400, width=900
).configure_view(strokeWidth=0)